# DeepVAR -- Deep Vector Autoregression (Pipeline B)

**Model**: PyTorch multivariate LSTM (DeepVAR equivalent, replaces `gluonts.mx`)
**Target**: `gdpc1`
**Variables**: Cat3 (53 + COVID = 56)
**Train**: 1959-2007 / **Val**: 2008-2016 / **Test**: 2017-2025
**Scaling**: YES (StandardScaler per variable before sequence construction)
**HP tuning**: NO (architecture fixed to match original: num_layers=4, num_cells=40, context_length=12)
**Note**: Original used `gluonts.mx.DeepVAREstimator` which requires MXNet (max Python 3.9, deprecated).
Replaced with an equivalent PyTorch LSTM: takes all Cat3 vars as multivariate input, predicts all vars
1 step ahead, extracts gdpc1. Same context_length=12, LSTM depth, and rolling-train logic.
**Runtime**: ~30-60s per vintage. Full test loop ~1-3 hours.

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = [15, 10]

sys.path.insert(0, os.path.join(os.path.pardir, "data"))
from helpers import (gen_lagged_data, mean_fill_dataset, get_features,
                     load_data, PREDICTIONS_DIR, TARGET, COVID)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

TRAIN_START    = "1959-01-01"
TRAIN_END      = "2007-12-31"
VAL_END        = "2016-12-31"
TEST_START     = "2017-01-01"
TEST_END       = "2025-12-31"
CONTEXT_LENGTH = 12   # matches original gluonts context_length
NUM_LAYERS     = 4    # matches original num_layers
NUM_CELLS      = 40   # matches original num_cells
DROPOUT        = 0.01 # matches original dropout_rate
EPOCHS         = 10   # matches original Trainer(epochs=10)
LR             = 1e-4 # matches original learning_rate
BATCH_SIZE     = 100  # matches original batch_size
VINTAGES       = {"m1": -2, "m2": -1, "m3": 0}

DEVICE = torch.device("cpu")

print("DeepVAR (PyTorch) -- Cat3, context_length={}".format(CONTEXT_LENGTH))


class DeepVARNet(nn.Module):
    """Multivariate LSTM: takes [B, T, n_vars] → predicts [B, n_vars] one step ahead."""
    def __init__(self, n_vars, hidden=40, n_layers=4, dropout=0.01):
        super().__init__()
        self.lstm = nn.LSTM(n_vars, hidden, n_layers, batch_first=True,
                            dropout=dropout if n_layers > 1 else 0.0)
        self.fc = nn.Linear(hidden, n_vars)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])


def make_sequences(arr, context_length):
    """Slide context window over T×n matrix; X=[B,T,n], y=[B,n] (next step)."""
    X, y = [], []
    for t in range(context_length, len(arr)):
        X.append(arr[t - context_length:t])
        y.append(arr[t])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)


def train_deepvar(arr_scaled, context_length, n_vars, epochs, lr, batch_size, seed):
    torch.manual_seed(seed)
    X, y = make_sequences(arr_scaled, context_length)
    if len(X) == 0:
        return None
    ds = torch.utils.data.TensorDataset(torch.tensor(X), torch.tensor(y))
    dl = torch.utils.data.DataLoader(ds, batch_size=batch_size, shuffle=True)
    model = DeepVARNet(n_vars, NUM_CELLS, NUM_LAYERS, DROPOUT).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    model.train()
    for _ in range(epochs):
        for xb, yb in dl:
            opt.zero_grad()
            loss_fn(model(xb.to(DEVICE)), yb.to(DEVICE)).backward()
            opt.step()
    return model


def predict_deepvar(model, arr_scaled, context_length):
    model.eval()
    ctx = torch.tensor(arr_scaled[-context_length:][np.newaxis], dtype=torch.float32)
    with torch.no_grad():
        pred_scaled = model(ctx.to(DEVICE)).cpu().numpy()[0]
    return pred_scaled

print("Model class defined.")

In [ ]:
monthly, _, metadata = load_data()
features = get_features("cat3", with_covid=True)
print("Features: {}".format(len(features)))

# DeepVAR is multivariate: predict TARGET + all Cat3 vars 1-step-ahead.
# gdpc1 is NOT in get_features() (it's the target), so add it explicitly.
all_vars   = [TARGET] + features   # gdpc1 at index 0, then Cat3
target_idx = 0
var_cols   = ["date"] + all_vars

test_quarters = monthly[(monthly["date"] >= TEST_START) &
                         (monthly["date"] <= TEST_END) &
                         monthly["date"].dt.month.isin([3, 6, 9, 12])]["date"].tolist()

predictions = {v: [] for v in VINTAGES}
actuals_list = []
failed = 0

for i, q_date in enumerate(test_quarters):
    pd_q = pd.Timestamp(q_date)
    actual = monthly[monthly["date"] == pd_q][TARGET].values[0]
    actuals_list.append(actual)

    fc_m3 = pd_q
    train = monthly[(monthly["date"] >= TRAIN_START) & (monthly["date"] <= fc_m3)].reset_index(drop=True)
    train = train[var_cols]
    train_m = gen_lagged_data(metadata, train.copy(), fc_m3, lag=0)
    train_m[TARGET] = train_m[TARGET].shift(1)   # 1-step-ahead target (same as original)
    train_filled = mean_fill_dataset(train_m, train_m)

    arr_train = train_filled[all_vars].values.astype(np.float32)
    scaler = StandardScaler()
    arr_scaled = scaler.fit_transform(arr_train)

    try:
        model = train_deepvar(arr_scaled, CONTEXT_LENGTH, len(all_vars),
                              EPOCHS, LR, BATCH_SIZE, SEED)
        if model is None:
            raise ValueError("Not enough data")
    except Exception:
        for vn in VINTAGES:
            predictions[vn].append(np.nan)
        failed += 3
        continue

    for vn, offset in VINTAGES.items():
        fc = pd_q + pd.DateOffset(months=offset)
        test_slice = monthly[(monthly["date"] >= TRAIN_START) & (monthly["date"] <= fc)].reset_index(drop=True)
        test_slice = test_slice[var_cols]
        test_m = gen_lagged_data(metadata, test_slice.copy(), fc, lag=0)
        test_filled = mean_fill_dataset(train_m, test_m)

        try:
            arr_test = test_filled[all_vars].values.astype(np.float32)
            arr_test_scaled = scaler.transform(arr_test)
            pred_scaled = predict_deepvar(model, arr_test_scaled, CONTEXT_LENGTH)
            pred_all = scaler.inverse_transform(pred_scaled[np.newaxis])[0]
            pred = pred_all[target_idx]
        except Exception:
            pred = np.nan
            failed += 1

        predictions[vn].append(pred)

    if (i + 1) % 4 == 0:
        print("  {} / {}".format(i + 1, len(test_quarters)))

print("Done. {} quarters, {} failures.".format(len(test_quarters), failed))

In [ ]:
os.makedirs(PREDICTIONS_DIR, exist_ok=True)
for vn in VINTAGES:
    pd.DataFrame({
        "date": test_quarters, "actual": actuals_list,
        "prediction": predictions[vn],
    }).to_csv(os.path.join(PREDICTIONS_DIR, "deepvar_{}.csv".format(vn)), index=False)
    print("Saved deepvar_{}.csv".format(vn))


In [ ]:
def rmse(a, p):
    m = ~(np.isnan(a) | np.isnan(p))
    return np.sqrt(np.mean((a[m] - p[m]) ** 2)) if m.sum() > 0 else np.nan

def mae(a, p):
    m = ~(np.isnan(a) | np.isnan(p))
    return np.mean(np.abs(a[m] - p[m])) if m.sum() > 0 else np.nan

panels = {
    "Pre-COVID  (2017-2019)": ("2017-01-01", "2019-12-31"),
    "COVID      (2020-2021)": ("2020-04-01", "2021-12-31"),
    "Post-COVID (2022-2025)": ("2022-01-01", "2025-12-31"),
    "Full test  (2017-2025)": ("2017-01-01", "2025-12-31"),
}
a = np.array(actuals_list)
d = pd.to_datetime(test_quarters)
for pn, (ps, pe) in panels.items():
    m = (d >= ps) & (d <= pe)
    print("--- {} ---".format(pn))
    for vn in VINTAGES:
        pp = np.array(predictions[vn])
        print("  {}  RMSFE={:.6f}  MAE={:.6f}  N={}".format(
            vn, rmse(a[m], pp[m]), mae(a[m], pp[m]), m.sum()))
